# Data Channel Visualization
Load cached `.npy` files from `cache/` and visually inspect each product group.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

CACHE_DIR = 'cache'
timestamps = pd.DatetimeIndex(np.load(os.path.join(CACHE_DIR, 'timestamps.npy'), allow_pickle=True))
print(f'Timestamps: {len(timestamps)}, {timestamps[0]} -> {timestamps[-1]}')

In [ ]:
def load(name):
    path = os.path.join(CACHE_DIR, f'{name}.npy')
    arr = np.load(path)
    nan_pct = np.isnan(arr).mean() * 100
    print(f'{name}: {arr.shape}, NaN: {nan_pct:.1f}%, range: [{np.nanmin(arr):.4g}, {np.nanmax(arr):.4g}]')
    return arr

def show_frames(data, titles, t_idx, suptitle=None, cmap='viridis'):
    """Show multiple channels side by side for a single timestep."""
    n = len(data)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, arr, title in zip(axes, data, titles):
        im = ax.imshow(arr[t_idx], cmap=cmap)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if suptitle:
        fig.suptitle(f'{suptitle}  |  {timestamps[t_idx]}', fontsize=13)
    plt.tight_layout()
    plt.show()

def show_coverage(data, titles, suptitle=None):
    """Show % non-NaN coverage over time for each channel."""
    fig, ax = plt.subplots(figsize=(12, 3))
    for arr, title in zip(data, titles):
        coverage = (~np.isnan(arr)).mean(axis=(1, 2)) * 100
        ax.plot(timestamps[:len(coverage)], coverage, label=title, linewidth=0.8)
    ax.set_ylabel('Coverage %')
    ax.set_ylim(-5, 105)
    ax.legend(fontsize=8)
    if suptitle:
        ax.set_title(f'{suptitle} — Data Coverage Ov"""  """er Time')
    plt.tight_layout()
    plt.show()

In [ ]:
# Pick a timestep to visualize, you can change this to whatever you want.
T_IDX = len(timestamps) // 2
print(f'Viewing timestep {T_IDX}: {timestamps[T_IDX]}')

---
## 1. GOES AOD / FRP / ADP

In [ ]:
goes_aod       = load('goes_aod')
goes_frp       = load('goes_frp')
goes_adp_smoke = load('goes_adp_smoke')
goes_adp_dust  = load('goes_adp_dust')

In [ ]:
show_frames(
    [goes_aod, goes_frp, goes_adp_smoke, goes_adp_dust],
    ['AOD', 'FRP Power', 'ADP Smoke', 'ADP Dust'],
    T_IDX, suptitle='GOES'
)

In [ ]:
show_coverage(
    [goes_aod, goes_frp, goes_adp_smoke, goes_adp_dust],
    ['AOD', 'FRP Power', 'ADP Smoke', 'ADP Dust'],
    suptitle='GOES'
)

---
## 2. HRRR Meteorological

In [ ]:
hrrr_temp   = load('hrrr_temp_2m')
hrrr_dew    = load('hrrr_dewpoint_2m')
hrrr_rh     = load('hrrr_rh_2m')
hrrr_psurf  = load('hrrr_pressure_surface')
hrrr_pmsl   = load('hrrr_pressure_msl')

In [ ]:
show_frames(
    [hrrr_temp, hrrr_dew, hrrr_rh, hrrr_psurf, hrrr_pmsl],
    ['Temp 2m', 'Dewpoint 2m', 'RH 2m', 'Pressure Sfc', 'Pressure MSL'],
    T_IDX, suptitle='HRRR — Thermodynamic'
)

In [ ]:
hrrr_pbl   = load('hrrr_pbl_height')
hrrr_u     = load('hrrr_u_wind')
hrrr_v     = load('hrrr_v_wind')
hrrr_smoke = load('hrrr_smoke_column')

In [ ]:
show_frames(
    [hrrr_pbl, hrrr_u, hrrr_v, hrrr_smoke],
    ['PBL Height', 'U Wind 10m', 'V Wind 10m', 'Smoke Column'],
    T_IDX, suptitle='HRRR — Dynamics & Smoke'
)

In [ ]:
show_coverage(
    [hrrr_temp, hrrr_dew, hrrr_rh, hrrr_psurf, hrrr_pmsl,
     hrrr_pbl, hrrr_u, hrrr_v, hrrr_smoke],
    ['Temp', 'Dewpt', 'RH', 'P_sfc', 'P_msl',
     'PBL', 'U', 'V', 'Smoke'],
    suptitle='HRRR'
)

---
## 3. MODIS NDVI

In [ ]:
ndvi = load('ndvi')

In [ ]:
show_frames([ndvi], ['NDVI'], T_IDX, suptitle='MODIS NDVI', cmap='YlGn')

In [ ]:
show_coverage([ndvi], ['NDVI'], suptitle='MODIS NDVI')

---
## 4. Static Fields (stubs)

In [ ]:
orog = load('orography')
lmask = load('land_mask')

In [ ]:
show_frames(
    [orog, lmask],
    ['Orography (stub)', 'Land Mask (stub)'],
    0, suptitle='Static Fields', cmap='terrain'
)

---
## 5. Assembled Tensor Sanity Check

In [ ]:
from pipeline import GapFillPipeline

pipe = GapFillPipeline(cache_dir=CACHE_DIR)
pipe.load()
pipe.summary()

In [ ]:
X, y = pipe.get_sample(T_IDX)
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'X NaN: {np.isnan(X).mean()*100:.1f}%')
print(f'y NaN: {np.isnan(y).mean()*100:.1f}%')

In [ ]:
# Show per-channel stats of the assembled tensor
fig, ax = plt.subplots(figsize=(14, 4))
channel_names = []
for ch in pipe.channels:
    for lag in ch.lags:
        channel_names.append(f'{ch.name}_t-{lag}')

means = [np.nanmean(X[i]) for i in range(X.shape[0])]
ax.bar(range(len(means)), means, width=0.8)
ax.set_xticks(range(len(means)))
ax.set_xticklabels(channel_names, rotation=90, fontsize=7)
ax.set_ylabel('Mean value')
ax.set_title(f'Per-channel mean at t={T_IDX} ({timestamps[T_IDX]})')
plt.tight_layout()
plt.show()